# Kharcha: which open-source OCR reads your bills best?
Compares four **open-source, self-hosted** OCR engines (all allowed by the AI-Forge-2026 rules: no paid or hosted APIs) on your 25 bills:
Tesseract (what the app uses now), EasyOCR, docTR and PaddleOCR.

Score = share of the real prices and totals (from the answer key) that each engine managed to read. Runs on the free Colab CPU in roughly 15-25 minutes. Use **GPU** (Runtime > Change runtime type) if you have it; it is faster, not required.

Each engine installs in its own cell. If one fails to install or run, the notebook prints the error and moves on, so one broken engine never blocks the rest.

## 1. Install

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-nep > /dev/null
!pip -q install pytesseract opencv-python-headless

In [ ]:
!pip -q install easyocr

In [ ]:
!pip -q install "python-doctr[torch]"

In [ ]:
!pip -q install paddlepaddle paddleocr

## 2. Answer key and photos

In [ ]:
%%writefile gold.py
# Hand-read answers. date = Gregorian ISO if printed, else None; bs = printed Bikram Sambat date.
# total = amount actually paid. items = (name, qty, amount, category)
GOLD = {
 "1.jpg": dict(merchant="Binayak Furniture", date=None, bs="2082-04-18", total=4802.50,
               items=[("Ply chair",1,4250.00,"Shopping")], dup_of=None, note="handwritten, pink paper"),
 "2.jpg": dict(merchant="Meridian International School", date="2026-06-19", bs=None, total=9200.00,
               items=[("Academic Resource fee",1,9200.00,"Bills")], note="school fee receipt"),
 "3.jpg": dict(merchant="Bhat-Bhateni Super Market (Maharajgunj)", date=None, bs="2083-06-23", total=None,
               items=None, note="long bill, photo ends at item 28, total not visible: needs review"),
 "4.jpg": dict(merchant="Bhat-Bhateni Super Market (Tangal)", date=None, bs="2083-06-12", total=125.00,
               items=[("Costa Brava",1,145.00,"Groceries")], note="discount 20"),
 "5.jpg": dict(merchant="Bhat-Bhateni Super Market (Tangal)", date=None, bs="2083-06-16", total=90.00,
               items=[("UC Chicken",1,90.00,"Groceries")]),
 "6.jpg": dict(merchant="Pepe Pizza Restaurant and Bar", date="2026-10-02", bs=None, total=555.00,
               items=[("Buff Steam Momo",3,491.15,"Dining")], note="VAT 63.85 extra; printed date is MM/DD/YYYY"),
 "7.jpg": dict(merchant="Big Mart", date="2026-09-29", bs=None, total=344.00,
               items=[("Century Nachoz Hot&Spicy",1,50.00,"Groceries"),("Britannia Fruit Cake Gobb",2,230.00,"Groceries"),
                      ("Nestle Milky Bar Moosha",2,64.00,"Groceries")]),
 "8.jpg": dict(merchant="G.R.G Bakery Pvt Ltd", date="2026-10-12", bs="2083-06-26", total=340.00,
               items=[("Blueberry cheesecake pastry",1,190.00,"Dining"),("Plain cheesecake pastry",1,150.00,"Dining")]),
}

GOLD.update({
 "9.jpg": dict(merchant="Krispy Krunchy Fried Chicken", date="2026-10-07", bs="2083-06-21", total=999.00,
               items=[("Wednesday Big Bucket",1,884.07,"Dining")], note="VAT 114.93 extra; photo cut at left edge"),
 "10.jpg": dict(merchant="Bhat-Bhateni Super Market (Maharajgunj)", date=None, bs="2083-06-23", total=1915.00,
               items=[("Regan Steel",4,1600.00,"Shopping"),("Guangbo Acr",1,315.00,"Shopping")]),
 "11.jpg": dict(merchant="Bhat-Bhateni Super Market (Maharajgunj)", date=None, bs="2083-06-23", total=3033.00,
               items=[("Fashion Bar",1,1435.00,"Shopping"),("Revlon Supe",1,1598.00,"Shopping")]),
 "12.jpg": dict(merchant="Bhat-Bhateni Super Market (Maharajgunj)", date=None, bs="2083-06-23", total=2635.00,
               items=[("Fashion Men",1,2635.00,"Shopping")], note="very faded"),
 "13.jpg": dict(merchant="Bhat-Bhateni Super Market (Tangal)", date=None, bs="2083-06-16", total=1404.60,
               items=[("Urban Chick",1,245.00,"Groceries"),("Urban Buff",2,400.00,"Groceries"),("Jabsons Pea",2,58.00,"Groceries"),
                      ("Cadbury Dai",4,100.00,"Groceries"),("Berrak Spag",1,180.00,"Groceries"),("Bahar Pasta",1,190.00,"Groceries"),
                      ("Himalaya Pu",1,144.00,"Shopping"),("Harbal Gaja",1,102.00,"Shopping")],
               note="discount 14.40, creased, bottom cut"),
 "14.jpg": dict(merchant="Pepe Pizza Restaurant and Bar", date="2026-10-02", bs=None, total=350.00,
               items=[("Margherita Pizza",1,309.73,"Dining")], note="VAT 40.27 extra"),
 "15.jpg": dict(merchant="Bhat-Bhateni Super Market (Tangal)", date=None, bs="2083-06-05", total=2085.00,
               items=[("Dhara Healt",1,341.00,"Groceries"),("Dhara Kachi",1,401.00,"Groceries"),("Tide + Extr",1,310.00,"Shopping"),
                      ("Nivea Soft",1,361.00,"Shopping"),("Kanisha Cot",1,55.00,"Shopping"),("STC Salt 1K",1,25.00,"Groceries"),
                      ("BBSM Besan",1,125.00,"Groceries"),("Organic Mil",1,146.00,"Groceries"),("Mangaldeep",1,100.00,"Shopping"),
                      ("Vim Bar 100",4,40.00,"Shopping"),("Tide White",2,96.00,"Shopping"),("Shagun Yell",1,85.00,"Groceries")]),
})

GOLD.update({
 "16.jpg": dict(merchant="Bhat-Bhateni Super Market (Boudha)", date=None, bs="2083-06-04", total=695.00,
               items=[("TY Clothes",1,305.00,"Shopping"),("Faber Caste",1,120.00,"Shopping"),("Note Book X",1,135.00,"Shopping"),
                      ("Guangbo Sup",1,40.00,"Shopping"),("Kriti Artis",1,95.00,"Shopping")]),
 "18.jpg": dict(merchant="Bhat-Bhateni Super Market (Tangal)", date=None, bs="2083-06-16", total=3648.00,
               items=[("Lady Half T",1,1729.00,"Shopping"),("Lotus Baby",1,1919.00,"Shopping")]),
 "19.jpg": dict(merchant="Bhat-Bhateni Super Market (Tangal)", date=None, bs="2083-06-16", total=1784.00,
               items=[("Lux Cozy Me",1,229.00,"Shopping"),("Zoom Plus M",1,1555.00,"Shopping")]),
 "20.jpg": dict(merchant="Big Mart", date="2026-09-13", bs=None, total=615.00,
               items=[("Khajurico Puff Regular Gr",1,70.00,"Groceries"),("Newari Shahi Pulao Basmat",1,225.00,"Groceries"),
                      ("Nestle Kitkat",4,80.00,"Groceries"),("Britannia 50-50 Time Pass",1,120.00,"Groceries"),
                      ("Britannia 50-50 Sweet & S",1,120.00,"Groceries")],
               note="total hidden by crease; customer paid 1000, refund 385 => 615"),
 "21.jpg": dict(merchant="New Mishra Rice Mill", date=None, bs="2083-06-12", total=2700.00,
               items=[("Rice",30,2700.00,"Groceries")], note="handwritten Nepali on pink paper; figures are my best reading"),
 "25.jpg": dict(merchant="Sun International Incorporated Pvt Ltd", date="2026-07-09", bs="2083-03-25", total=1205.00,
               items=[("Regular Scoop",1,345.13,"Dining"),("Regular Scoop",1,345.13,"Dining"),("Regular Scoop",1,376.11,"Dining")],
               note="VAT 138.63 extra; printed 07/09/2026 is MM/DD (matches Miti 25/03/2083)"),
})
# photos of the same paper bill: keep each pair on the same side of the train/test split
DUPLICATES = {"24.jpg": "1.jpg", "23.jpg": "2.jpg", "22.jpg": "3.jpg", "17.jpg": "16.jpg"}

In [ ]:
import os, shutil, glob
from google.colab import files
os.makedirs("photos", exist_ok=True)
up = files.upload()                      # choose Bills.zip
for name, data in up.items():
    open(name, "wb").write(data)
    if name.lower().endswith(".zip"):
        !unzip -q -o "{name}" -d unzipped
        for p in glob.glob("unzipped/**/*.*", recursive=True):
            if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                shutil.copy(p, os.path.join("photos", os.path.basename(p)))
print(len(os.listdir("photos")), "photos")

## 3. Scoring helpers

In [ ]:
import re, time, json
import numpy as np
from PIL import Image, ImageOps
from gold import GOLD

def load(name, max_side=2200):
    im = ImageOps.exif_transpose(Image.open(f"photos/{name}")).convert("RGB")
    s = max_side / max(im.size)
    return im.resize((round(im.width * s), round(im.height * s))) if s < 1 else im

def group_boxes(items):
    """items: (x0, y0, x1, y1, text, conf) boxes -> lines of text, top to bottom, left to right."""
    items = sorted(items, key=lambda b: (b[1] + b[3]) / 2)
    rows = []
    for b in items:
        cy, h = (b[1] + b[3]) / 2, max(b[3] - b[1], 1)
        if rows and abs(cy - rows[-1]["cy"]) < 0.6 * rows[-1]["h"]:
            rows[-1]["w"].append(b); n = len(rows[-1]["w"])
            rows[-1]["cy"] += (cy - rows[-1]["cy"]) / n; rows[-1]["h"] = max(rows[-1]["h"], h)
        else:
            rows.append({"cy": cy, "h": h, "w": [b]})
    out = []
    for r in rows:
        ws = sorted(r["w"], key=lambda b: b[0])
        out.append((" ".join(w[4] for w in ws), float(np.mean([w[5] for w in ws]))))
    return out

def numbers(text):
    return set(x.replace(",", "") for x in re.findall(r"\d[\d,]*\.\d{1,2}|\d[\d,]*", text))

def score(lines, g):
    """(prices found, prices wanted, total found?) against the answer key."""
    nums = numbers(" ".join(t for t, _ in lines))
    def has(v):
        s = f"{v:.2f}"
        return s in nums or (s.endswith(".00") and s[:-3] in nums) or (s.endswith("0") and s[:-1] in nums)
    want = [i[2] for i in (g["items"] or [])]
    hit = sum(has(v) for v in want)
    tot = g["total"] is not None and has(g["total"])
    return hit + tot, len(want) + (g["total"] is not None), tot

# quick self-test of the helpers (no OCR needed)
fake = [(0,0,50,10,"Net",90),(60,1,120,11,"Amount:",80),(0,40,50,50,"90.00",95)]
assert [t for t, _ in group_boxes(fake)] == ["Net Amount:", "90.00"]
assert score([("Net Amount 90.00", 90)], {"items": [("x", 1, 90.0, "")], "total": 90.0}) == (2, 2, True)
print("helpers ok")

## 4. The four engines
Each returns `[(line_text, confidence 0-100), ...]` for a photo.

In [ ]:
import pytesseract, cv2

def tesseract(name):
    im = load(name, 2200)
    g = np.array(im.convert("L"))
    bg = cv2.GaussianBlur(g, (0, 0), max(10, g.shape[1] // 36))
    g = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.divide(g, bg, scale=255))
    d = pytesseract.image_to_data(Image.fromarray(g), lang="eng+nep", config="--oem 1 --psm 6", output_type=pytesseract.Output.DICT)
    items = [(d["left"][i], d["top"][i], d["left"][i] + d["width"][i], d["top"][i] + d["height"][i], d["text"][i], float(d["conf"][i]))
             for i in range(len(d["text"])) if d["text"][i].strip() and float(d["conf"][i]) > 0]
    return group_boxes(items)

ENGINES = {"tesseract": tesseract}

In [ ]:
try:
    import easyocr
    _easy = easyocr.Reader(["en", "ne"], gpu=False)      # use gpu=True if you switched to a GPU runtime
    def easy(name):
        res = _easy.readtext(np.array(load(name)), paragraph=False)
        items = [(min(p[0] for p in b), min(p[1] for p in b), max(p[0] for p in b), max(p[1] for p in b), t, c * 100) for b, t, c in res]
        return group_boxes(items)
    ENGINES["easyocr"] = easy
except Exception as e:
    print("EasyOCR unavailable:", repr(e)[:300])

In [ ]:
try:
    from doctr.io import DocumentFile
    from doctr.models import ocr_predictor
    _doctr = ocr_predictor(det_arch="db_resnet50", reco_arch="crnn_vgg16_bn", pretrained=True)
    def doctr(name):
        out = _doctr(DocumentFile.from_images([f"photos/{name}"]))
        lines = []
        for page in out.pages:
            for block in page.blocks:
                for ln in block.lines:
                    y = np.mean([w.geometry[0][1] + w.geometry[1][1] for w in ln.words]) / 2
                    lines.append((y, " ".join(w.value for w in ln.words), 100 * float(np.mean([w.confidence for w in ln.words]))))
        return [(t, c) for _, t, c in sorted(lines)]
    ENGINES["doctr"] = doctr
except Exception as e:
    print("docTR unavailable:", repr(e)[:300])

In [ ]:
try:
    from paddleocr import PaddleOCR
    try:   # PaddleOCR 3.x
        _paddle = PaddleOCR(lang="en", use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
    except TypeError:   # PaddleOCR 2.x
        _paddle = PaddleOCR(lang="en", use_angle_cls=False, show_log=False)
    def paddle(name):
        path = f"photos/{name}"
        res = _paddle.predict(path) if hasattr(_paddle, "predict") else _paddle.ocr(path)
        items = []
        if isinstance(res, list) and res and isinstance(res[0], dict) or (res and hasattr(res[0], "get")):
            r = res[0]
            for poly, t, c in zip(r.get("rec_polys", r.get("dt_polys")), r["rec_texts"], r["rec_scores"]):
                p = np.array(poly); items.append((p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max(), t, float(c) * 100))
        else:
            for poly, (t, c) in res[0]:
                p = np.array(poly); items.append((p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max(), t, float(c) * 100))
        return group_boxes(items)
    ENGINES["paddleocr"] = paddle
except Exception as e:
    print("PaddleOCR unavailable:", repr(e)[:300])
print("engines ready:", list(ENGINES))

## 5. Run them all on your bills
Bills 3 and 22 (cut off) are skipped; they have no answer. This takes a while: leave it running.

In [ ]:
names = [n for n in sorted(GOLD, key=lambda x: (len(x), x)) if GOLD[n]["items"] is not None]
results = {e: {} for e in ENGINES}
for e, fn in ENGINES.items():
    t0 = time.time()
    for n in names:
        try:
            lines = fn(n)
            results[e][n] = {"lines": lines, "score": score(lines, GOLD[n])[:2], "total_found": score(lines, GOLD[n])[2]}
        except Exception as ex:
            print(e, n, "failed:", repr(ex)[:150])
    print(e, "done in", round(time.time() - t0), "s", flush=True)
json.dump(results, open("ocr_results.json", "w"), ensure_ascii=False)

In [ ]:
print(f"{'engine':11}{'prices+totals found':>22}{'totals found':>15}{'avg word conf':>15}")
for e, r in results.items():
    H = sum(v["score"][0] for v in r.values()); N = sum(v["score"][1] for v in r.values())
    T = sum(v["total_found"] for v in r.values())
    conf = np.mean([c for v in r.values() for _, c in v["lines"]]) if r else 0
    print(f"{e:11}{H:>8}/{N:<4}={H/max(N,1):>5.0%}{T:>10}/{len(r):<4}{conf:>12.0f}")
print()
print("per bill (prices found / wanted)")
print("bill".ljust(8) + "".join(e.ljust(12) for e in results))
for n in names:
    print(n.ljust(8) + "".join((f"{results[e][n]['score'][0]}/{results[e][n]['score'][1]}" if n in results[e] else "-").ljust(12) for e in results))

## 6. Send me the result
Run the next cell, download `ocr_results.json` and upload it in our chat. I will read the winner's actual output on your bills, wire it into the reader, and re-measure confidence.

In [ ]:
files.download("ocr_results.json")

### Reading the table
* **prices+totals found** is the number that matters. The old setup read about two thirds.
* **avg word conf** is each engine's own opinion of itself. It is not comparable between engines, so do not choose by it.
* An engine that wins by a few points on 21 bills is a tie. Look for a clear gap.
* All four are open-source and run on your own machine, which is what the Frogtoberfest rules ask for.